In [1]:
%pip install -q "transformers>=4.46,<5" datasets accelerate sentencepiece sacrebleu jiwer


import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import json
import random
import unicodedata
from pathlib import Path

import jiwer
import numpy as np
import sacrebleu
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Compute device: cpu
PyTorch: 2.14.1+cpu


In [2]:
DATA_DIR = Path("data")
SYNTHETIC_DIR = DATA_DIR / "synthetic"
MODEL_DIR = Path("models") / "mt5_small_telugu_gec"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = SYNTHETIC_DIR / "train.json"
VAL_FILE = SYNTHETIC_DIR / "val.json"
TEST_FILE = SYNTHETIC_DIR / "test.json"

for data_file in (TRAIN_FILE, VAL_FILE, TEST_FILE):
    if not data_file.exists():
        raise FileNotFoundError(f"Required project data file not found: {data_file.resolve()}")

def load_pairs(path):
    with path.open("r", encoding="utf-8") as file:
        rows = json.load(file)
    pairs = []
    for row in rows:
        source = unicodedata.normalize("NFC", str(row.get("input", ""))).strip()
        target = unicodedata.normalize("NFC", str(row.get("target", ""))).strip()
        if source and target:
            pairs.append({"input": source, "target": target})
    return pairs

train_pairs = load_pairs(TRAIN_FILE)
val_pairs = load_pairs(VAL_FILE)
test_pairs = load_pairs(TEST_FILE)
print(f"Training pairs: {len(train_pairs):,}")
print(f"Validation pairs: {len(val_pairs):,}")
print(f"Test pairs: {len(test_pairs):,}")
print(f"Training pairs with a changed input: {sum(p['input'] != p['target'] for p in train_pairs):,}")
print(train_pairs[:3])

Training pairs: 60,000
Validation pairs: 7,500
Test pairs: 7,500
Training pairs with a changed input: 47,536
[{'input': 'ఎర్రనకు రెండు బిరుదులున్నాయి శంభుదాసుడుప్రబంధ పరమేశ్వరుడు', 'target': 'ఎర్రనకు రెండు బిరుదులున్నాయి శంభుదాసుడు ప్రబంధ పరమేశ్వరుడు'}, {'input': 'కమాంట్ లాంగ్వేజ్, ఇతర ప్రోగ్రామ్ లను ప్రారంభించడం వంటి కంప్యూటర్ యొక్క విధులను నియంత్రించడనాికి ఉపయోగించే భాష', 'target': 'కమాండ్ లాంగ్వేజ్, ఇతర ప్రోగ్రామ్ లను ప్రారంభించడం వంటి కంప్యూటర్ యొక్క విధులను నియంత్రించడానికి ఉపయోగించే భాష'}, {'input': 'శాశ్వత ప్చచిక ప్రాంతాలు, ఇతర మేత భూమిః 91 హెక్టార్లు', 'target': 'శాశ్వత పచ్చిక ప్రాంతాలు, ఇతర మేత భూమిః 91 హెక్టార్లు'}]


In [3]:
BASE_MODEL = "google/mt5-small"
SOURCE_PREFIX = "correct this: [TE] "
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

def tokenize_batch(batch):
    model_inputs = tokenizer(
        [SOURCE_PREFIX + text for text in batch["input"]],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = Dataset.from_list(train_pairs).map(
    tokenize_batch, batched=True, remove_columns=["input", "target"]
)
val_dataset = Dataset.from_list(val_pairs).map(
    tokenize_batch, batched=True, remove_columns=["input", "target"]
)
test_dataset = Dataset.from_list(test_pairs).map(
    tokenize_batch, batched=True, remove_columns=["input", "target"]
)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)
print(train_dataset)
print(val_dataset)
print(test_dataset)

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

c:\Users\leela\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\leela\.cache\huggingface\hub\models--google--mt5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
c:\Users\leela\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


... (3 outputs left out)

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Map:   0%|          | 0/7500 [00:00<?, ? examples/s]

Map:   0%|          | 0/7500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 60000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7500
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7500
})


In [4]:
def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_references = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_predictions = [text.strip() for text in decoded_predictions]
    decoded_references = [text.strip() for text in decoded_references]
    return {
        "gleu": sacrebleu.corpus_bleu(decoded_predictions, [decoded_references]).score,
        "wer": jiwer.wer(decoded_references, decoded_predictions),
        "cer": jiwer.cer(decoded_references, decoded_predictions),
        "exact_match": float(np.mean([p == r for p, r in zip(decoded_predictions, decoded_references)])),
    }

training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.05,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=1000,
    logging_strategy="epoch",
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("Best checkpoint selection metric: validation GLEU")

Best checkpoint selection metric: validation GLEU


In [5]:
checkpoint_dir = MODEL_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
latest_checkpoint = get_last_checkpoint(str(checkpoint_dir))

if latest_checkpoint:
    print(f"Resuming training from: {latest_checkpoint}")
    train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("No checkpoint found. Starting training from the pretrained base model.")
    train_result = trainer.train()

trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()
print(f"Completed model and tokenizer saved to: {MODEL_DIR.resolve()}")

c:\Users\leela\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)
trainer.log_metrics("test", test_metrics)
trainer.save_metrics("test", test_metrics)
test_metrics

In [ ]:
def correct_telugu(text, num_beams=4, max_new_tokens=128):
    model_inputs = tokenizer(
        SOURCE_PREFIX + text.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
    ).to(saved_model.device)
    output_ids = saved_model.generate(
        **model_inputs,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        early_stopping=True,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

saved_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
saved_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(DEVICE)
saved_model.eval()
tokenizer = saved_tokenizer

for pair in test_pairs[:10]:
    print(f"Noisy : {pair['input']}")
    print(f"Output: {correct_telugu(pair['input'])}")
    print(f"Target: {pair['target']}\n")